# Solutions · 10 · Dispersive waves and spectral methods

Worked solutions to the exercises of [notebook 10](../notebooks/10_dispersive_waves_and_spectral_methods.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, spectral
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Integrate the KdV soliton with an integrating factor: write v = exp(-D3 t) u in Fourier space so that the linear term disappears, and integrate only the nonlinear term explicitly. How much larger can the step be?

In [2]:
p = pr.kdv_soliton(); n, L = 256, 40.0
k = spectral.wavenumbers(n, L)
E = -(1j * k) ** 3                                           # the linear operator -u_xxx in Fourier space: multiply by -(ik)^3
def rhs_if(t, vhat_ri):                                     # integrating factor: v = exp(-E t) u_hat; v' = exp(-E t) N(u)
    vhat = vhat_ri[:n] + 1j * vhat_ri[n:]
    uhat = np.exp(E * t) * vhat
    u = np.fft.ifft(uhat).real
    N = np.fft.fft(-6 * u * spectral.derivative(u, L, 1))
    d = np.exp(-E * t) * N
    return np.r_[d.real, d.imag]
v0 = np.fft.fft(p.y0)
s = ti.dopri45(rhs_if, np.r_[v0.real, v0.imag], (0, 4.0), 1e-8, 1e-10, n_out=1)
u_end = np.fft.ifft(np.exp(E * 4.0) * (s.y[-1][:n] + 1j * s.y[-1][n:])).real
print(f"integrating factor: {s.steps} steps (mean h {np.mean(s.extra['h']):.1e}) vs {p.solve('dopri', rtol=1e-8, atol=1e-10).steps} without; error {np.max(np.abs(u_end - p.exact(p.x, 4.0))):.1e}")

integrating factor: 3248 steps (mean h 1.2e-03) vs 30172 without; error 4.8e-11


Solving the linear dispersive part exactly by the integrating factor removes its stiffness from the explicit
integrator, which now sees only the nonlinear term: the step is set by the soliton's own time scale and the
step count drops by an order of magnitude at the same accuracy. (The exponential grows with t and is better
re-centred every step in a production code; here t ≤ 4 and it is harmless.)

## Exercise 2

Give the NLS soliton twice the amplitude (2 sech(2x) e^(4it) is also a soliton). Measure the step the adaptive integrator needs and explain it from the frequency 4.

In [3]:
n, L = 256, 40.0
x = spectral.grid(-L / 2, L / 2, n)
for A in (1.0, 2.0):
    psi0 = A / np.cosh(A * x) + 0j                             # A sech(Ax) exp(i A^2 t) is a soliton of i psi_t + psi_xx + 2|psi|^2 psi = 0
    def rhs(t, y):
        psi = y[:n] + 1j * y[n:]
        d = 1j * (spectral.derivative(psi, L, 2) + 2 * np.abs(psi) ** 2 * psi)
        return np.r_[d.real, d.imag]
    s = ti.dopri45(rhs, np.r_[psi0.real, psi0.imag], (0, 2.0), 1e-8, 1e-10, n_out=1)
    psi = s.y[-1][:n] + 1j * s.y[-1][n:]
    print(f"amplitude {A}: {s.steps} steps; error vs A sech(Ax) e^(i A^2 t) {np.max(np.abs(psi - psi0 * np.exp(1j * A**2 * 2.0))):.1e}")

amplitude 1.0: 524 steps; error vs A sech(Ax) e^(i A^2 t) 2.1e-08


amplitude 2.0: 582 steps; error vs A sech(Ax) e^(i A^2 t) 1.4e-07


The amplitude-2 soliton oscillates at frequency A² = 4, four times faster, and its width is half - yet the
integrator needs only about 10 % more steps. The reason is that the step was already set by the spectral
stiffness of ψ_xx (eigenvalues up to k_max² ≈ 400, the same in both runs) rather than by the solution's own
frequency; the faster soliton merely raises the accuracy requirement a little. Only when the solution's
frequency approaches the stiffness-limited rate does the step count rise in proportion.

## Exercise 3

**Implement it yourself:** write the KdV right-hand side with the 2/3 dealiasing applied to the product u u_x, and compare the two-soliton invariants with and without it at n = 64 (under-resolved).

In [4]:
for dealiased in (False, True):
    for n in (64, 256):
        p = pr.kdv_two_solitons(n=n); L = 40.0
        def rhs(t, u, dealiased=dealiased):
            ux = spectral.derivative(u, L, 1)
            prod = spectral.dealias(spectral.dealias(u) * spectral.dealias(ux)) if dealiased else u * ux
            return -6 * prod - spectral.derivative(u, L, 3)
        with np.errstate(all="ignore"):
            s = ti.dopri45(rhs, p.y0, (0, 8.0), 1e-8, 1e-10, n_out=1, max_steps=200000)
        mom = p.grid.integrate(s.y[-1] ** 2) - p.params["momentum"] if np.all(np.isfinite(s.y[-1])) else np.nan
        print(f"{'dealiased' if dealiased else 'plain    '} n = {n:3d}: reached t = {s.t[-1]:.1f}, momentum drift {mom:+.2e}, max |u| {np.max(np.abs(s.y[-1])):.3g}")

plain     n =  64: reached t = 7.0, momentum drift +2.03e+306, max |u| 1.28e+153


plain     n = 256: reached t = 8.0, momentum drift +1.00e-13, max |u| 2


dealiased n =  64: reached t = 8.0, momentum drift -3.32e-09, max |u| 1.86


dealiased n = 256: reached t = 8.0, momentum drift -9.77e-15, max |u| 2


At n = 64 the plain product aliases enough energy into the resolved modes to blow the run up; the 2/3-rule
product keeps it stable, though with 64 points the solitons are still poorly resolved and the invariants
drift. At n = 256 both are exact to rounding. Dealiasing removes a *spurious* instability; it does not supply
resolution.